# Importing the required libraries

In [17]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import ttest_ind
import math
import csv




In [18]:

def load_csv_dataset(filepath):
    with open(filepath, newline='') as csvfile:
        data_reader = csv.reader(csvfile)
        headers = next(data_reader)  # Assuming the first row contains headers
        data = [row for row in data_reader]
    return data, headers


In [19]:
def manual_encode_feature(feature_values):

    unique_values = sorted(set(feature_values))
    value_to_int = {value: i for i, value in enumerate(unique_values)}
    return [value_to_int[value] for value in feature_values]

def preprocess_dataset(dataset):

    # Transpose the dataset for easier column-wise processing
    transposed_dataset = list(zip(*dataset))

    # Process each feature column
    X = []
    for feature_values in transposed_dataset[:-1]:  # Exclude the last column (target variable)
        encoded_features = manual_encode_feature(feature_values)
        X.append(encoded_features)

    # Process the target variable (last column)
    y = manual_encode_feature(transposed_dataset[-1])

    # Transpose X back to the original row-wise format
    X = list(zip(*X))

    return X, y





In [20]:
def sklearn_knn(X, y, n_neighbors=5):
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    scores = cross_val_score(clf, X, y, cv=10)  # 10-fold cross-validation
    return scores.mean(),scores

In [21]:

'''
def euclidean_distance(a, b):
    return math.sqrt(sum((e1 - e2) ** 2 for e1, e2 in zip(a, b)))
'''
def weighted_vote(neighbors):
    class_weights = {}
    for distance, class_label in neighbors:
        if distance == 0:
            weight = float('inf')
        else:
            weight = 1 / distance
        if class_label in class_weights:
            class_weights[class_label] += weight
        else:
            class_weights[class_label] = weight
    return max(class_weights, key=class_weights.get)

def break_ties(neighbors, outcomes):
    max_count = 0
    closest_class = None
    for class_label in set(outcomes):
        count = outcomes.count(class_label)
        if count > max_count or (count == max_count and closest_class is None):
            max_count = count
            closest_class = class_label
    return closest_class

def custom_knn(X_train, y_train, X_test, k=5):
    predictions = []
    for test_point in X_test:
        distances = [(euclidean_distance(test_point, X_train[i]), y_train[i]) for i in range(len(X_train))]
        distances.sort(key=lambda x: x[0])
        neighbors = distances[:k]
        outcomes = [neighbor[1] for neighbor in neighbors]

        # Weighted voting
        prediction = weighted_vote(neighbors)

        # Tie-breaking
        unique_outcomes = set(outcomes)
        if outcomes.count(prediction) > 1 and len(unique_outcomes) > 1:
            prediction = break_ties(neighbors, outcomes)

        predictions.append(prediction)
    return predictions


In [22]:

def cross_validate(X, y, k=10):
    fold_size = len(X) // k
    accuracies = []

    for fold in range(k):
        # Manually split the dataset into training and testing parts
        X_train = X[:fold * fold_size] + X[(fold + 1) * fold_size:]
        y_train = y[:fold * fold_size] + y[(fold + 1) * fold_size:]
        X_test = X[fold * fold_size:(fold + 1) * fold_size]
        y_test = y[fold * fold_size:(fold + 1) * fold_size]



        # Use the custom KNN function to predict test set
        y_pred = custom_knn(X_train, y_train, X_test, k=5)

        # Calculate accuracy
        correct_predictions = sum(1 for actual, predicted in zip(y_test, y_pred) if actual == predicted)
        accuracy = correct_predictions / len(y_test)
        accuracies.append(accuracy)

    # Calculate average accuracy across all folds
    average_accuracy = sum(accuracies) / k
    return average_accuracy, accuracies


In [23]:


def perform_ttest(scores1, scores2):
    t_stat, p_val = ttest_ind(scores1, scores2)
    return t_stat, p_val



# Breast Cancer

In [24]:
data, meta = load_csv_dataset('breast_cancer.csv')

X, y = preprocess_dataset(data)
print( sklearn_knn(X, y))
sklearn_knn_accuracy,cancer_list1 = sklearn_knn(X, y)
print("Scikit-Learn KNN Accuracy: ", sklearn_knn_accuracy)

(0.7168719211822661, array([0.65517241, 0.68965517, 0.75862069, 0.75862069, 0.68965517,
       0.72413793, 0.82142857, 0.67857143, 0.67857143, 0.71428571]))
Scikit-Learn KNN Accuracy:  0.7168719211822661


In [25]:

cv_accuracy,cancer_list2 = cross_validate(X, y)
print("Accuracies of breastcancer data set",cancer_list2)
print(f"Average 10-fold CV Accuracy: {cv_accuracy}")


Accuracies of breastcancer data set [0.75, 0.7142857142857143, 0.6428571428571429, 0.6785714285714286, 0.6785714285714286, 0.7142857142857143, 0.6785714285714286, 0.6428571428571429, 0.6785714285714286, 0.7142857142857143]
Average 10-fold CV Accuracy: 0.6892857142857144


In [26]:

t_stat, p_val = perform_ttest(cancer_list1, cancer_list2)
print("T-test statistic: ", t_stat, "P-value: ", p_val)
# Decide whether to accept or reject the null hypothesis
if p_val < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

T-test statistic:  1.4399595935375122 P-value:  0.16704536159478892
Fail to reject the null hypothesis.


# Car Evaluation

In [27]:
data, meta = load_csv_dataset('car_evaluation.csv')



X, y = preprocess_dataset(data)
print( sklearn_knn(X, y))
sklearn_knn_accuracy,car_list1 = sklearn_knn(X, y)
print("Scikit-Learn KNN Accuracy: ", sklearn_knn_accuracy)

(0.7448346551955909, array([0.57225434, 0.67052023, 0.70520231, 0.71098266, 0.76878613,
       0.72254335, 0.73410405, 0.81395349, 0.88372093, 0.86627907]))
Scikit-Learn KNN Accuracy:  0.7448346551955909


In [28]:

cv_accuracy,car_list2 = cross_validate(X, y)
print("Accuracies of car evalution data set",car_list2)
print(f"Average 10-fold CV Accuracy: {cv_accuracy}")

Accuracies of car evalution data set [0.7732558139534884, 0.7267441860465116, 0.7732558139534884, 0.9709302325581395, 0.8255813953488372, 0.7267441860465116, 0.7616279069767442, 0.686046511627907, 0.8372093023255814, 0.6104651162790697]
Average 10-fold CV Accuracy: 0.7691860465116279


In [29]:

t_stat, p_val = perform_ttest(car_list1, car_list2)
print("T-test statistic: ", t_stat, "P-value: ", p_val)
# Decide whether to accept or reject the null hypothesis
if p_val < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

T-test statistic:  -0.5733836262938034 P-value:  0.5734796418424004
Fail to reject the null hypothesis.


# Hayes Roth

In [30]:
data, meta = load_csv_dataset('hayes-roth.data.csv')



print( sklearn_knn(X, y))
sklearn_knn_accuracy,hayes_list1 = sklearn_knn(X, y)
print("Scikit-Learn KNN Accuracy: ", sklearn_knn_accuracy)

(0.7448346551955909, array([0.57225434, 0.67052023, 0.70520231, 0.71098266, 0.76878613,
       0.72254335, 0.73410405, 0.81395349, 0.88372093, 0.86627907]))
Scikit-Learn KNN Accuracy:  0.7448346551955909


In [31]:
cv_accuracy,hayes_list2 = cross_validate(X, y)
print("Accuracies of hayes roth data set",hayes_list2)
print(f"Average 10-fold CV Accuracy: {cv_accuracy}")

Accuracies of hayes roth data set [0.7732558139534884, 0.7267441860465116, 0.7732558139534884, 0.9709302325581395, 0.8255813953488372, 0.7267441860465116, 0.7616279069767442, 0.686046511627907, 0.8372093023255814, 0.6104651162790697]
Average 10-fold CV Accuracy: 0.7691860465116279


In [32]:

t_stat, p_val = perform_ttest(hayes_list1, hayes_list2)
print("T-test statistic: ", t_stat, "P-value: ", p_val)
# Decide whether to accept or reject the null hypothesis
if p_val < 0.05:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

T-test statistic:  -0.5733836262938034 P-value:  0.5734796418424004
Fail to reject the null hypothesis.
